# RIMES Preprocessing — Step by Step

Questo notebook esegue la pipeline di preprocessing RIMES **uno step alla volta**,
salvando e visualizzando il risultato intermedio ad ogni passaggio.

A differenza di IAM (un form intero -> un'immagine), RIMES parte da **singole righe di
testo** che vengono processate individualmente e poi **impilate verticalmente** in
batch da 5 righe per formare canvas 448x448 per ogni writer.

Pipeline:
1. Caricamento di alcune righe originali di uno stesso writer
2. Per ciascuna riga: binarizzazione Otsu
3. Per ciascuna riga: rimozione righe nere top/bottom
4. Per ciascuna riga: tight crop (proiezioni assiali, rho=0.018)
5. Split delle righe troppo larghe (>896px)
6. Calcolo dell'altezza minima del writer (clip [20,32])
7. Resize di ogni riga all'altezza comune, centratura in canvas 32x448 (max)
8. Batching: stack verticale di gruppi da 5 righe in canvas 448x448

Ad ogni step salviamo i PNG intermedi in `./steps_output_rimes/`.


## 0. Setup e configurazione

In [ ]:
import cv2
import os
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict

# ==================== CONFIGURAZIONE ====================
# Cartella con le immagini RIMES originali (singole righe)
RIMES_INPUT = "../datasets/RIMES/Handwritten2Text Training Dataset/Images"  # <-- MODIFICA

# Writer da usare per la demo. Se None, viene scelto automaticamente il primo
# writer che ha almeno MIN_LINES_DEMO righe disponibili.
WRITER_ID = None
MIN_LINES_DEMO = 6   # quante righe di questo writer usare per la demo di batching

OUTPUT_DIR = "steps_output_rimes"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TARGET_SIZE = 448
LINE_HEIGHT = 32
LINES_PER_BATCH = TARGET_SIZE // LINE_HEIGHT - 9  # 448 // 32 - 9 = 5
MAX_WIDTH = TARGET_SIZE * 2  # 896, soglia di split

def show_and_save(img, title, filename, cmap="gray", figsize=(12, 4)):
    """Salva l'immagine intermedia su disco e la mostra inline"""
    out_path = os.path.join(OUTPUT_DIR, filename)
    cv2.imwrite(out_path, img)
    plt.figure(figsize=figsize)
    plt.imshow(img, cmap=cmap)
    plt.title(f"{title}\nshape: {img.shape}  |  salvato in: {out_path}")
    plt.axis("off")
    plt.show()
    print(f"Salvato: {out_path}  |  shape: {img.shape}")


## 1. Scansione dataset e selezione di un writer

Raggruppiamo i file per writer (come nel notebook originale, parsing del nome file)
e selezioniamo un writer con abbastanza righe per mostrare bene il batching.


In [ ]:
writer_images = defaultdict(list)

for img_name in sorted(os.listdir(RIMES_INPUT)):
    if not img_name.endswith((".jpg", ".png", ".jpeg")):
        continue
    parts = img_name.split("-")
    if len(parts) < 2:
        continue
    writer_id = parts[1].split("_")[0]
    writer_images[writer_id].append(os.path.join(RIMES_INPUT, img_name))

print(f"Trovati {len(writer_images)} writer totali")

if WRITER_ID is None:
    # sceglie il primo writer con abbastanza righe
    for wid, paths in writer_images.items():
        if len(paths) >= MIN_LINES_DEMO:
            WRITER_ID = wid
            break
    assert WRITER_ID is not None, "Nessun writer con abbastanza righe trovato, riduci MIN_LINES_DEMO"

demo_paths = sorted(writer_images[WRITER_ID])[:MIN_LINES_DEMO]
print(f"Writer selezionato per la demo: {WRITER_ID}  |  righe usate: {len(demo_paths)}")
for p in demo_paths:
    print("  -", os.path.basename(p))


## 2. Visualizza le righe originali

Prima di processarle, guardiamo come sono le righe grezze.


In [ ]:
fig, axes = plt.subplots(len(demo_paths), 1, figsize=(12, 2 * len(demo_paths)))
if len(demo_paths) == 1:
    axes = [axes]
for ax, p in zip(axes, demo_paths):
    img = cv2.imread(p)
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(os.path.basename(p))
    ax.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "01_original_lines.png"), dpi=150, bbox_inches="tight")
plt.show()


## 3. Preprocessing di una singola riga (dettagliato passo-passo)

Prendiamo la **prima riga** del writer e mostriamo ogni singolo step nel dettaglio.
Le altre righe verranno poi processate con la stessa funzione, in modo compatto,
nella sezione successiva (per costruire il batch).


In [ ]:
def apply_threshold(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if len(image.shape) == 3 else image
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return gray, binary

demo_line_path = demo_paths[0]
image = cv2.imread(demo_line_path)

gray, binary = apply_threshold(image)

show_and_save(gray, "03a - Grayscale", "03a_grayscale.png")
show_and_save(binary, "03b - Binary (Otsu)", "03b_binary.png")


### 3.1 Rimozione righe nere top/bottom

In [ ]:
def compute_top_bottom_black_rows_crop(binary_image, black_ratio_threshold=0.5):
    h, w = binary_image.shape
    black_pixels_per_row = np.sum(binary_image == 0, axis=1)
    black_ratio = black_pixels_per_row / w
    valid_rows = np.where(black_ratio <= black_ratio_threshold)[0]
    if len(valid_rows) == 0:
        return 0, h
    return valid_rows[0], valid_rows[-1] + 1

first_row, last_row = compute_top_bottom_black_rows_crop(binary, black_ratio_threshold=0.5)
print(f"first_row={first_row}, last_row={last_row}  (h originale={binary.shape[0]})")

gray_cleaned = gray[first_row:last_row, :]
binary_cleaned = binary[first_row:last_row, :]

show_and_save(gray_cleaned, "03c - Dopo rimozione righe nere (grayscale)", "03c_gray_cleaned.png")
show_and_save(binary_cleaned, "03d - Dopo rimozione righe nere (binary)", "03d_binary_cleaned.png")


### 3.2 Tight crop (proiezioni assiali, rho=0.018)

In [ ]:
def compute_tight_crop_coords(binary_image, padding=5, min_pixel_ratio=0.018):
    binary_inv = 255 - binary_image
    h, w = binary_image.shape

    min_row_threshold = max(10, int(w * min_pixel_ratio))
    min_col_threshold = max(10, int(h * min_pixel_ratio))

    row_sums = np.sum(binary_inv > 0, axis=1)
    col_sums = np.sum(binary_inv > 0, axis=0)

    text_rows = np.where(row_sums > min_row_threshold)[0]
    text_cols = np.where(col_sums > min_col_threshold)[0]

    if len(text_rows) == 0 or len(text_cols) == 0:
        return None

    y1, y2 = text_rows[0], text_rows[-1]
    x1, x2 = text_cols[0], text_cols[-1]

    extra_trim = 3
    y1 = min(y1 + extra_trim, y2 - 1)
    y2 = max(y2 - extra_trim, y1 + 1)

    y1 = max(0, y1 - padding)
    y2 = min(h, y2 + padding)
    x1 = max(0, x1 - padding)
    x2 = min(w, x2 + padding)

    return (y1, y2, x1, x2)

crop_coords = compute_tight_crop_coords(binary_cleaned, padding=5)  # rho default = 0.018
assert crop_coords is not None
y1, y2, x1, x2 = crop_coords
print(f"Tight crop coords: y1={y1}, y2={y2}, x1={x1}, x2={x2}")

vis = cv2.cvtColor(binary_cleaned.copy(), cv2.COLOR_GRAY2BGR)
cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 0, 255), 2)
show_and_save(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB), "03e - Bounding box tight crop (su binary_cleaned)", "03e_bbox.png", cmap=None)

gray_cropped = gray_cleaned[y1:y2, x1:x2]
show_and_save(gray_cropped, "03f - Risultato tight crop (grayscale)", "03f_gray_cropped.png")

print(f"\nDimensione riga dopo tight crop: {gray_cropped.shape}")
if gray_cropped.shape[1] > MAX_WIDTH:
    print(f"Riga più larga di {MAX_WIDTH}px -> verrebbe splittata a metà")
else:
    print(f"Riga entro il limite di {MAX_WIDTH}px -> nessuno split necessario")


## 4. Preprocessing compatto di TUTTE le righe del writer

Ora applichiamo la stessa funzione (`preprocess_rimes_line`, che include anche
lo split delle righe troppo lunghe) a tutte le righe selezionate del writer,
per poter costruire il batch.


In [ ]:
def preprocess_rimes_line(image, max_width=MAX_WIDTH):
    gray, binary = apply_threshold(image)

    first_row, last_row = compute_top_bottom_black_rows_crop(binary, black_ratio_threshold=0.5)
    gray_cleaned = gray[first_row:last_row, :]
    binary_cleaned = binary[first_row:last_row, :]

    crop_coords = compute_tight_crop_coords(binary_cleaned, padding=5)
    if crop_coords is None:
        return []

    y1, y2, x1, x2 = crop_coords
    gray_cropped = gray_cleaned[y1:y2, x1:x2]

    if gray_cropped.size == 0:
        return []

    if gray_cropped.shape[1] > max_width:
        mid = gray_cropped.shape[1] // 2
        return [gray_cropped[:, :mid], gray_cropped[:, mid:]]

    return [gray_cropped]

cropped_lines = []
for p in demo_paths:
    img = cv2.imread(p)
    if img is not None:
        line_splits = preprocess_rimes_line(img)
        cropped_lines.extend(line_splits)

print(f"Totale righe (dopo eventuale split) pronte per il batch: {len(cropped_lines)}")

fig, axes = plt.subplots(len(cropped_lines), 1, figsize=(12, 1.5 * len(cropped_lines)))
if len(cropped_lines) == 1:
    axes = [axes]
for ax, line in zip(axes, cropped_lines):
    ax.imshow(line, cmap="gray")
    ax.set_title(f"shape={line.shape}")
    ax.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "04_all_cropped_lines.png"), dpi=150, bbox_inches="tight")
plt.show()


## 5. Calcolo altezza minima del writer (clip [20, 32])

$$h_{\min} = \text{clip}\left(\min_l h_l,\ 20,\ 32\right)$$


In [ ]:
min_height = min(line.shape[0] for line in cropped_lines)
print(f"Altezza minima grezza tra le righe: {min_height}")

min_height = max(min_height, 20)
min_height = min(min_height, LINE_HEIGHT)  # 32

print(f"h_min dopo clip [20,32]: {min_height}")


## 6. Resize di ogni riga a h_min, centratura in canvas (h_min x 448)

Ogni riga viene ridimensionata mantenendo l'aspect ratio (larghezza massima 448),
poi centrata verticalmente (e allineata a sinistra orizzontalmente) in un canvas
bianco di dimensioni fisse `h_min x 448`.


In [ ]:
def resize_to_fixed_height(image, target_height, target_width):
    h, w = image.shape
    new_w = int(w * target_height / h)

    if new_w > target_width:
        new_w = target_width
        new_h = int(h * target_width / w)
    else:
        new_h = target_height

    resized = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_AREA)

    canvas = np.full((target_height, target_width), 255, dtype=np.uint8)
    y_offset = (target_height - new_h) // 2
    canvas[y_offset:y_offset + new_h, :new_w] = resized

    return canvas

lines_normalized = []
for line in cropped_lines:
    resized_line = resize_to_fixed_height(line, min_height, TARGET_SIZE)
    lines_normalized.append(resized_line)

assert all(l.shape[0] == min_height for l in lines_normalized), "Altezze non uniformi!"

fig, axes = plt.subplots(len(lines_normalized), 1, figsize=(12, 1.2 * len(lines_normalized)))
if len(lines_normalized) == 1:
    axes = [axes]
for ax, line in zip(axes, lines_normalized):
    ax.imshow(line, cmap="gray")
    ax.set_title(f"shape={line.shape}  (canvas {min_height}x{TARGET_SIZE})")
    ax.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "06_normalized_lines.png"), dpi=150, bbox_inches="tight")
plt.show()


## 7. Batching: stack verticale a gruppi di 5 righe

$$
N_{\text{batch}} = \lfloor 448/32 \rfloor - 9 = 5
$$

Le righe height-normalized vengono impilate verticalmente in gruppi da 5;
il canvas finale è sempre 448x448, con eventuale padding bianco se il writer
non ha abbastanza righe per riempire l'ultimo batch.


In [ ]:
print(f"LINES_PER_BATCH = {LINES_PER_BATCH}")

batches = []
for i in range(0, len(lines_normalized), LINES_PER_BATCH):
    batch_lines = lines_normalized[i:i + LINES_PER_BATCH]
    batch = np.vstack(batch_lines)

    canvas = np.full((TARGET_SIZE, TARGET_SIZE), 255, dtype=np.uint8)
    canvas[:batch.shape[0], :] = batch
    batches.append(canvas)

print(f"Numero di batch prodotti per il writer {WRITER_ID}: {len(batches)}")
print(f"Ogni batch occupa al massimo {LINES_PER_BATCH * min_height}px di altezza su 448 disponibili "
      f"(il resto e\' padding bianco)")

for idx, b in enumerate(batches):
    show_and_save(b, f"07 - Batch {idx} (writer {WRITER_ID})", f"07_batch_{idx:02d}.png", figsize=(6, 6))


## 8. Riepilogo visivo

In [ ]:
fig, axes = plt.subplots(1, len(batches) + 1, figsize=(6 * (len(batches) + 1), 6))
if len(batches) == 0:
    axes = [axes]

orig_grid_path = os.path.join(OUTPUT_DIR, "01_original_lines.png")
if os.path.exists(orig_grid_path) and len(batches) >= 1:
    axes = np.atleast_1d(axes)

for idx, b in enumerate(batches):
    ax = axes[idx] if len(batches) > 1 else axes[0] if len(batches) == 1 else axes
    ax.imshow(b, cmap="gray")
    ax.set_title(f"Batch {idx}")
    ax.axis("off")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "00_summary_batches.png"), dpi=150, bbox_inches="tight")
plt.show()

print(f"\nPipeline completata. Tutti gli step intermedi sono in: {OUTPUT_DIR}/")
